<a href="https://colab.research.google.com/github/judithwang021/judith-6070/blob/main/WeeklyModules/Week05/Assignment5_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5: Improved FAANG Stock Data Notebook

This version keeps the original GitHub connection workflow and improves the repeated data-reading, combining, cleaning, and saving steps.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.formula.api as sm
import statsmodels.api as se
import seaborn as sns
import datetime as dt
from sklearn.linear_model import LinearRegression
from scipy import stats
from io import StringIO
import itertools
from pathlib import Path

pd.set_eng_float_format(accuracy=3, use_eng_prefix=True)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

from google.colab import userdata
import os

github_token = userdata.get('github_token')
owner = 'judithwang021' # Replace with the GitHub repository owner
repository = 'judith-6070' # Replace with the GitHub repository name

clone_url = f'https://{github_token}@github.com/{owner}/{repository}.git'

# Clone the repository
import subprocess
subprocess.run(['git', 'clone', clone_url])

# Navigate into the cloned repository directory (optional, but often useful)
os.chdir(repository)
print(f"Changed directoGit_HUBry to: {os.getcwd()}")


## Generative AI recommendations

- Store ticker symbols and file paths in one dictionary instead of repeating five `read_csv()` statements.
- Use a loop to read each file and add its ticker column consistently.
- Concatenate the data once with `ignore_index=True`.
- Clean and sort the combined data before saving it.
- Avoid saving a CSV and immediately reading the identical CSV back into Python.
- Use clear variable names and show useful results at each stage.

In [ ]:
# Store the GitHub data-file paths
stock_files = {
    'AAPL': Path('/content/judith-6070/data/aapl.csv'),
    'AMZN': Path('/content/judith-6070/data/amzn.csv'),
    'NFLX': Path('/content/judith-6070/data/nflx.csv'),
    'FB': Path('/content/judith-6070/data/fb.csv'),
    'GOOG': Path('/content/judith-6070/data/goog.csv')
}

# Read each stock file and add its ticker
stock_dataframes = []
for ticker, file_path in stock_files.items():
    stock_df = pd.read_csv(file_path)
    stock_df['ticker'] = ticker
    stock_dataframes.append(stock_df)

# Combine all five stocks into one dataframe
stock = pd.concat(stock_dataframes, ignore_index=True)
stock.head()


In [ ]:
# Clean and sort the combined data
stock['date'] = pd.to_datetime(stock['date'])
stock['volume'] = pd.to_numeric(stock['volume'], errors='coerce')
stock = stock.dropna(subset=['date', 'volume']).copy()
stock['volume'] = stock['volume'].astype(int)
stock = stock.sort_values(by=['date', 'ticker']).reset_index(drop=True)

stock.head()


In [ ]:
# Save the cleaned combined data once
output_path = Path('/content/judith-6070/data/faanf.csv')
stock.to_csv(output_path, index=False)
print(f'Saved combined data to: {output_path}')


In [ ]:
# Find the seven highest-volume trading days
top7 = stock.nlargest(7, 'volume')
top7[['date', 'ticker', 'volume']]


In [ ]:
# Convert the data from wide to long format
stock_long = stock.melt(
    id_vars=['date', 'ticker'],
    var_name='variable',
    value_name='value'
)

stock_long.head(10)
